# LeetCode 421 — Maximum XOR of Two Numbers in an Array bằng Python

Link đề: https://leetcode.com/problems/maximum-xor-of-two-numbers-in-an-array/ · Độ khó: **Medium** · Chủ đề: Bit Manipulation, Trie, Hash Table, Greedy

Skeleton LeetCode:

```python
class Solution:
    def findMaximumXOR(self, nums: List[int]) -> int:
        ...
```

Ghi chú: phần phát biểu đề dịch sang tiếng Việt từ đề gốc; các ví dụ đều được chạy lại bằng code trong notebook.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: trong mảng số nguyên không âm `nums`, tìm giá trị lớn nhất của `nums[i] XOR nums[j]` với `0 <= i <= j < n` (cho phép `i == j`).
2. Nắm công cụ Python/bit: toán tử `^` (XOR), `>>` (dịch phải), `&` (AND), `<<` (dịch trái), `bin(x)`, `format(x, "05b")`, cách lấy **bit thứ `i`** bằng `x >> i & 1`, tập hợp `set` và biểu thức sinh (`any`, `{... for ...}`).
3. Đi từ **vét cạn mọi cặp `O(n²)`** → **tham lam từng bit từ cao xuống thấp + tập tiền tố (hash set)** → **Trie nhị phân** → **chia để trị theo nhóm bit**.
4. Hiểu mấu chốt: bit cao có trọng số lớn hơn **tất cả bit thấp cộng lại**, nên ta cứ cố làm bit cao nhất của đáp án bằng `1` trước; và tính chất `a ^ b = c ⇔ a ^ c = b` biến việc "tìm cặp" thành "tra cứu trong tập hợp".
5. Tránh các lỗi kinh điển: duyệt thiếu bit (`range(30)` thay vì 31 bit), quên `ans = ans << 1`, trong Trie chọn **cùng** bit thay vì bit **đối lập**, và nhầm khi `n = 1` (đáp án `0`).

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Trả về **kết quả lớn nhất** của `nums[i] XOR nums[j]`, với `0 <= i <= j < n`.

**Ví dụ 1:** `nums = [3,10,5,25,2,8]` → `28`. Giải thích: lớn nhất là `5 XOR 25 = 28`.

**Ví dụ 2:** `nums = [14,70,53,83,49,91,36,80,92,51,66,70]` → `127`.

**Ràng buộc:** `1 <= nums.length <= 2 * 10^5`, `0 <= nums[i] <= 2^31 - 1`.

XOR là gì? `a XOR b` so sánh hai số **theo từng bit**: bit kết quả bằng `1` nếu hai bit **khác nhau**, bằng `0` nếu **giống nhau**.

In [1]:
from typing import List
import random
import time

examples = [
    ([3, 10, 5, 25, 2, 8], 28),
    ([14, 70, 53, 83, 49, 91, 36, 80, 92, 51, 66, 70], 127),
]
for nums, expected in examples:
    print(nums, "-> kỳ vọng", expected)

[3, 10, 5, 25, 2, 8] -> kỳ vọng 28
[14, 70, 53, 83, 49, 91, 36, 80, 92, 51, 66, 70] -> kỳ vọng 127


### 2.1. Đọc kỹ đề: vài điểm cần để ý

- **`i <= j` (có thể `i == j`):** `x XOR x = 0`, nên với `n = 1` đáp án là `0`. Cho phép `i == j` còn khiến đáp án **không bao giờ âm**.
- **Kích thước:** `n` tới `2 * 10^5` ⇒ vét cạn `n²/2 ≈ 2 * 10^10` cặp là không thể. Cần khoảng `O(n * 31)`.
- **Số lớn:** `nums[i] <= 2^31 - 1` nên cần **31 bit** (bit `0` đến bit `30`). Quên bit 30 sẽ sai ở các số lớn.

In [2]:
print("x ^ x =", 25 ^ 25)
print("2**31 - 1 =", 2**31 - 1, "-> nhị phân có", (2**31 - 1).bit_length(), "bit")
print("5 ^ 25 =", 5 ^ 25)

x ^ x = 0
2**31 - 1 = 2147483647 -> nhị phân có 31 bit
5 ^ 25 = 28


## 3. Kiến thức nền

### 3.1. Biểu diễn nhị phân: `bin` và `format`

`bin(x)` cho chuỗi dạng `'0b101'`; `format(x, "08b")` cho chuỗi nhị phân **đủ 8 ký tự**, thêm số `0` ở đầu — tiện để căn thẳng hàng các bit.

In [3]:
for x in [3, 10, 5, 25, 2, 8]:
    print(f"{x:>2} = {bin(x):>8} = {format(x, '05b')}")

 3 =     0b11 = 00011
10 =   0b1010 = 01010
 5 =    0b101 = 00101
25 =  0b11001 = 11001
 2 =     0b10 = 00010
 8 =   0b1000 = 01000


### 3.2. XOR `^` — bảng chân trị và tính chất

| a | b | a ^ b |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

Tính chất cần nhớ:

- `a ^ a = 0`, `a ^ 0 = a`.
- Giao hoán: `a ^ b = b ^ a`; kết hợp: `(a ^ b) ^ c = a ^ (b ^ c)`.
- **Đảo vế:** nếu `a ^ b = c` thì `a ^ c = b` (XOR cả hai vế với `a`). Tính chất này là xương sống của Cách 2.

In [4]:
a, b = 5, 25
c = a ^ b
print(format(a, "05b"), "^", format(b, "05b"), "=", format(c, "05b"), "=", c)
print("a ^ c =", a ^ c, "(đúng bằng b =", b, ")")
print("b ^ c =", b ^ c, "(đúng bằng a =", a, ")")

00101 ^ 11001 = 11100 = 28
a ^ c = 25 (đúng bằng b = 25 )
b ^ c = 5 (đúng bằng a = 5 )


### 3.3. Dịch bit `>>`, `<<` và lấy bit thứ `i`

- `x >> k`: bỏ `k` bit thấp nhất (chia nguyên cho `2^k`). Kết quả là **tiền tố** gồm các bit cao của `x`.
- `x << k`: thêm `k` bit `0` vào cuối (nhân `2^k`).
- `x & 1`: bit thấp nhất. Vậy bit thứ `i` của `x` là `x >> i & 1` (Python: `>>` ưu tiên cao hơn `&`).

In [5]:
x = 25            # 11001
for i in range(4, -1, -1):
    print(f"bit {i} của {x} = {x >> i & 1}   | tiền tố x >> {i} = {x >> i} = {bin(x >> i)}")
print("1 << 3 =", 1 << 3, "| 3 << 2 =", 3 << 2)

bit 4 của 25 = 1   | tiền tố x >> 4 = 1 = 0b1
bit 3 của 25 = 1   | tiền tố x >> 3 = 3 = 0b11
bit 2 của 25 = 0   | tiền tố x >> 2 = 6 = 0b110
bit 1 của 25 = 0   | tiền tố x >> 1 = 12 = 0b1100
bit 0 của 25 = 1   | tiền tố x >> 0 = 25 = 0b11001
1 << 3 = 8 | 3 << 2 = 12


### 3.4. `set` và `any` / biểu thức sinh

`set` lưu các phần tử không trùng, tra cứu `in` trung bình `O(1)`. `any(biểu thức for ... in ...)` trả `True` nếu **ít nhất một** phần tử thỏa. `{x >> 3 for x in nums}` tạo nhanh tập các tiền tố.

In [6]:
nums = [3, 10, 5, 25, 2, 8]
prefixes = {x >> 3 for x in nums}          # chỉ giữ bit 4 và 3 của mỗi số
print(prefixes)
print(any((p ^ 3) in prefixes for p in prefixes))     # có cặp p, q với p ^ q = 3 không?

{0, 1, 3}
True


### 3.5. Trie nhị phân: cây mà mỗi nút có tối đa 2 con (`0` và `1`)

Một **Trie** nhị phân lưu các số bằng cách đi từ bit **cao nhất xuống thấp nhất**; mỗi bit chọn nhánh con `0` hoặc `1`. Hai số có chung tiền tố bit thì chia sẻ đường đi. Với `nums = [2, 3]` (nhị phân 10, 11) ta được đường: gốc → `1` → `0` / `1`. Ta sẽ cài đặt thật ở Cách 3; ở đây chỉ làm một cây `dict` nhỏ để cảm nhận.

In [7]:
def build_dict_trie(nums, top):
    root = {}
    for x in nums:
        node = root
        for i in range(top, -1, -1):
            node = node.setdefault(x >> i & 1, {})
    return root


import json
print(json.dumps(build_dict_trie([2, 3], 1)))     # số 2 = 10, số 3 = 11: gốc -> 1 -> {0, 1}
print(json.dumps(build_dict_trie([5, 25], 4)))

{"1": {"0": {}, "1": {}}}
{"0": {"0": {"1": {"0": {"1": {}}}}}, "1": {"1": {"0": {"0": {"1": {}}}}}}


## 4. Ý tưởng nền tảng: tính tay một ví dụ

Với `nums = [3, 10, 5, 25, 2, 8]`, viết các số dưới dạng nhị phân 5 bit:

```
 3 = 00011
10 = 01010
 5 = 00101
25 = 11001
 2 = 00010
 8 = 01000
```

Ta muốn hai số **khác nhau ở càng nhiều bit càng tốt, đặc biệt ở các bit cao**. Chỉ `25` có bit cao nhất (bit 4) bằng `1`, các số còn lại có bit 4 bằng `0`. Vậy để XOR có bit 4 bằng `1`, một trong hai số phải là `25`. Ghép `25` lần lượt với từng số khác rồi so sánh.

In [8]:
nums = [3, 10, 5, 25, 2, 8]
for y in nums:
    print(f"25 ^ {y:>2}: {format(25, '05b')} ^ {format(y, '05b')} = {format(25 ^ y, '05b')} = {25 ^ y}")

25 ^  3: 11001 ^ 00011 = 11010 = 26
25 ^ 10: 11001 ^ 01010 = 10011 = 19
25 ^  5: 11001 ^ 00101 = 11100 = 28
25 ^ 25: 11001 ^ 11001 = 00000 = 0
25 ^  2: 11001 ^ 00010 = 11011 = 27
25 ^  8: 11001 ^ 01000 = 10001 = 17


Kết quả lớn nhất là `25 ^ 5 = 28 = 11100`. Nhận xét: sau khi bit 4 đã là `1`, ta tìm tiếp số có bit 3 **đối** với bit 3 của `25` (`25` có bit 3 = `1` nên cần bit 3 = `0`), cứ thế. Đây chính là **tham lam theo bit** và là ý tưởng của Cách 2/3.

## 5. Cách 1 — Brute Force: thử mọi cặp

**Ý tưởng:** hai vòng lặp `i <= j`, tính `nums[i] ^ nums[j]`, giữ giá trị lớn nhất. Khởi tạo `best = 0` (hợp lệ vì `x ^ x = 0` luôn có mặt do cho phép `i == j`).

In [9]:
def max_xor_brute(nums):
    n = len(nums)
    best = 0
    for i in range(n):
        for j in range(i, n):
            best = max(best, nums[i] ^ nums[j])
    return best


for nums, expected in examples:
    got = max_xor_brute(nums)
    print(nums[:6], "->", got, "| kỳ vọng", expected)
    assert got == expected

[3, 10, 5, 25, 2, 8] -> 28 | kỳ vọng 28
[14, 70, 53, 83, 49, 91] -> 127 | kỳ vọng 127


**Độ phức tạp:** `n(n+1)/2` cặp, mỗi cặp `O(1)` ⇒ **Time `O(n²)`**, Space `O(1)`.

**Nút thắt:** với `n = 2 * 10^5` ta có `2 * 10^10` cặp. Hãy đếm thử.

In [10]:
for n in [10**3, 10**4, 2 * 10**5]:
    print(f"n = {n:>7}: số cặp = {n * (n + 1) // 2:,}")
rng = random.Random(1)
arr = [rng.randrange(2**31) for _ in range(2000)]
t = time.perf_counter()
max_xor_brute(arr)
print("brute với n = 2000: %.2f giây (n = 2*10^5 sẽ chậm ~10^4 lần)" % (time.perf_counter() - t))

n =    1000: số cặp = 500,500
n =   10000: số cặp = 50,005,000
n =  200000: số cặp = 20,000,100,000
brute với n = 2000: 0.13 giây (n = 2*10^5 sẽ chậm ~10^4 lần)


## 6. Ý tưởng thuật toán / bước đột phá

**Quan sát 1 — bit cao thắng.** Giá trị của bit `i` là `2^i`, và `2^i > 2^(i-1) + ... + 2^0`. Do đó **một đáp án có bit cao hơn bằng `1` luôn lớn hơn** mọi đáp án thiếu bit đó, bất kể các bit thấp. ⇒ Quyết định từng bit của đáp án **từ cao xuống thấp**: bit nào có thể là `1` thì cứ chọn `1`.

**Quan sát 2 — kiểm tra "có thể là 1" bằng tra cứu.** Giả sử ta đã xác định các bit cao của đáp án là `ans` (tiền tố). Ta muốn thử `cand = ans` với bit hiện tại bằng `1`. `cand` đạt được ⇔ tồn tại hai số `a, b` trong mảng sao cho **tiền tố** (các bit từ cao tới bit hiện tại) của `a ^ b` bằng `cand`. Vì `XOR` của hai tiền tố chính là tiền tố của XOR, điều kiện là: tồn tại hai tiền tố `p, q` trong tập tiền tố sao cho `p ^ q = cand`. Theo tính chất đảo vế, `p ^ q = cand ⇔ q = p ^ cand`. Vậy chỉ cần với mỗi `p`, tra xem `p ^ cand` có trong tập không.

Ta kiểm chứng bằng tay trên ví dụ 1, chỉ xét hai bit cao nhất (bit 4 và bit 3):

In [11]:
nums = [3, 10, 5, 25, 2, 8]
# bit 4: ans trước = 0, thử cand = 1
P4 = {x >> 4 for x in nums}
print("tiền tố bit 4:", P4, "-> cand = 1:", any((p ^ 1) in P4 for p in P4))     # True: 0 ^ 1 = 1 (số 0 và số 1)
ans = 1
# bit 3: ans << 1 = 2, thử cand = 3
P3 = {x >> 3 for x in nums}
print("tiền tố bit 3:", P3, "-> cand = 3:", any((p ^ 3) in P3 for p in P3))

tiền tố bit 4: {0, 1} -> cand = 1: True
tiền tố bit 3: {0, 1, 3} -> cand = 3: True


Ở bit 3: tiền tố `{0, 1, 3}`; có `0 ^ 3 = 3` ✓ (`0` là tiền tố của `3, 5, 2`; `3 = 11` là tiền tố của `25`), nên `cand = 3` đạt được ⇒ `ans = 3 = 11` (hai bit cao của đáp án là `11`). Tiếp tục tới bit 0 ta được `11100 = 28`. Mục 8 sẽ trace đầy đủ.

## 7. Các cách tiếp theo

### Cách 2 — Tham lam từng bit + tập tiền tố (hash set), `O(31 * n)`

**Giải thích từng đoạn:**

1. `ans = 0`. Với `i` từ `30` xuống `0`:
2. `prefixes = {x >> i for x in nums}` — các tiền tố gồm bit `30..i`.
3. `ans <<= 1` — chừa chỗ cho bit mới (đáp án tới bit `i` có thêm một bit cuối).
4. `cand = ans | 1` — giả sử bit mới là `1`.
5. Nếu `any((p ^ cand) in prefixes for p in prefixes)` thì `ans = cand`.
6. Cuối cùng `ans` là đáp án.

Lưu ý: nhờ `ans <<= 1` mỗi vòng nên `ans` luôn là **tiền tố đáp án** có cùng độ dài với tiền tố trong `prefixes`.

In [12]:
def max_xor_prefix_set(nums, top=30):
    ans = 0
    for i in range(top, -1, -1):
        prefixes = {x >> i for x in nums}
        ans <<= 1
        cand = ans | 1
        if any((p ^ cand) in prefixes for p in prefixes):
            ans = cand
    return ans


for nums, expected in examples:
    got = max_xor_prefix_set(nums)
    print(got, "| kỳ vọng", expected)
    assert got == expected

28 | kỳ vọng 28
127 | kỳ vọng 127


In [13]:
# Code nộp LeetCode — Cách 2
class Solution2:
    def findMaximumXOR(self, nums: List[int]) -> int:
        ans = 0
        for i in range(30, -1, -1):
            prefixes = {x >> i for x in nums}
            ans <<= 1
            cand = ans | 1
            if any((p ^ cand) in prefixes for p in prefixes):
                ans = cand
        return ans


print(Solution2().findMaximumXOR([3, 10, 5, 25, 2, 8]))   # 28

28


**Độ phức tạp:** 31 vòng, mỗi vòng tạo tập `O(n)` và kiểm tra `O(n)` ⇒ **Time `O(31 * n)`**, Space `O(n)`.

**Tối ưu nhỏ:** vì `x ^ y = cand` có tính đối xứng nên không cần loại trừ trường hợp `p == q`; nếu `cand` khác `0` thì `p ^ cand != p`.

### Cách 3 — Trie nhị phân (tìm bạn đối lập tốt nhất cho mỗi số)

**Ý tưởng:** xây Trie chứa các số đã duyệt. Với số mới `x`, đi từ bit cao xuống thấp: ở mỗi bit ta muốn bit đối lập (`1 - bit`) để XOR ra `1`; nếu nhánh đối lập tồn tại thì đi theo và cộng `1 << i` vào kết quả; nếu không, đành đi theo nhánh cùng bit (XOR ra `0`).

**Giải thích từng đoạn:**

1. Nút `Trie` có `children = [None, None]`.
2. `insert(x)`: đi từ bit 30 xuống 0, tạo nút nếu thiếu.
3. `search(x)`: đi cùng cách nhưng ưu tiên nhánh `v ^ 1`.
4. Với mỗi số `x`: `ans = max(ans, trie.search(x))`, rồi `insert(x)` (để cặp `i == j` hợp lệ ta có thể insert trước search; ở đây insert rồi search cho đơn giản).

In [14]:
class Trie:
    def __init__(self):
        self.children = [None, None]

    def insert(self, x, top=30):
        node = self
        for i in range(top, -1, -1):
            v = x >> i & 1
            if node.children[v] is None:
                node.children[v] = Trie()
            node = node.children[v]

    def search(self, x, top=30):
        node = self
        res = 0
        for i in range(top, -1, -1):
            v = x >> i & 1
            if node.children[v ^ 1] is not None:      # có bạn đối lập -> bit này của XOR là 1
                res |= 1 << i
                node = node.children[v ^ 1]
            else:                                     # đành đi cùng nhánh -> bit XOR là 0
                node = node.children[v]
        return res


def max_xor_trie(nums, top=30):
    trie = Trie()
    ans = 0
    for x in nums:
        trie.insert(x, top)
        ans = max(ans, trie.search(x, top))
    return ans


for nums, expected in examples:
    got = max_xor_trie(nums)
    print(got, "| kỳ vọng", expected)
    assert got == expected

28 | kỳ vọng 28
127 | kỳ vọng 127


Vì sao insert trước rồi search được? Sau khi `insert(x)` thì Trie đã chứa `x`, nên `search(x)` có thể gặp chính `x` (cho XOR `0`) — hợp lệ vì đề cho `i == j`; và vì mọi cặp `(i, j)` với `i < j` được xét khi ta xử lý phần tử đến sau, nên không bỏ sót cặp nào.

In [15]:
# Code nộp LeetCode — Cách 3
class Trie3:
    __slots__ = ("children",)

    def __init__(self):
        self.children: list = [None, None]

    def insert(self, x: int) -> None:
        node = self
        for i in range(30, -1, -1):
            v = x >> i & 1
            if node.children[v] is None:
                node.children[v] = Trie3()
            node = node.children[v]

    def search(self, x: int) -> int:
        node = self
        res = 0
        for i in range(30, -1, -1):
            v = x >> i & 1
            if node.children[v ^ 1] is not None:
                res |= 1 << i
                node = node.children[v ^ 1]
            else:
                node = node.children[v]
        return res


class Solution3:
    def findMaximumXOR(self, nums: List[int]) -> int:
        trie = Trie3()
        ans = 0
        for x in nums:
            trie.insert(x)
            ans = max(ans, trie.search(x))
        return ans


print(Solution3().findMaximumXOR([14, 70, 53, 83, 49, 91, 36, 80, 92, 51, 66, 70]))   # 127

127


**Độ phức tạp:** mỗi `insert`/`search` đi 31 bước ⇒ **Time `O(31 * n)`**; Space `O(31 * n)` nút trong xấu nhất (mỗi số có thể tạo đường riêng).

**So với Cách 2:** cùng bậc thời gian; Cách 3 trả lời được truy vấn "tìm số cho XOR lớn nhất với `x` bất kỳ" (dùng lại ở nhiều bài khác), còn Cách 2 gọn và tốn ít bộ nhớ hơn.

### Cách 4 — Chia để trị theo nhóm bit (hai nhóm A, B)

**Ý tưởng khác hẳn:** định nghĩa `best(A, B, bit)` = giá trị XOR lớn nhất của `a ^ b` với `a ∈ A`, `b ∈ B`, chỉ xét các bit từ `bit` xuống `0`. Tách `A` thành `A0` (bit `bit` bằng 0) và `A1` (bằng 1), tương tự `B0`, `B1`.

- Nếu có thể ghép **khác bit** (`A0` với `B1` hoặc `A1` với `B0` đều không rỗng): bit `bit` của đáp án chắc chắn là `1`; đáp án là `(1 << bit) + max(best(A0, B1, bit-1), best(A1, B0, bit-1))` (với nhánh nào không rỗng).
- Ngược lại (chỉ ghép được **cùng bit**): bit này của XOR bằng `0`; đáp án là `max(best(A0, B0, bit-1), best(A1, B1, bit-1))` (nhánh nào không rỗng).
- `bit < 0`: trả `0`.

Gọi `best(nums, nums, 30)`. Mỗi số tham gia vào tối đa vài cuộc gọi ở mỗi tầng ⇒ tổng công việc `O(31 * n)`.

In [16]:
def max_xor_divide(nums, top=30):
    def best(A, B, bit):
        if not A or not B:
            return -1                         # nhánh rỗng: không có cặp nào
        if bit < 0:
            return 0
        A0 = [x for x in A if not x >> bit & 1]
        A1 = [x for x in A if x >> bit & 1]
        B0 = [x for x in B if not x >> bit & 1]
        B1 = [x for x in B if x >> bit & 1]
        cross = max(best(A0, B1, bit - 1), best(A1, B0, bit - 1))
        if cross >= 0:                        # ghép khác bit được -> bit này của XOR là 1
            return (1 << bit) + cross
        return max(best(A0, B0, bit - 1), best(A1, B1, bit - 1))
    return best(nums, nums, top)


for nums, expected in examples:
    got = max_xor_divide(nums)
    print(got, "| kỳ vọng", expected)
    assert got == expected

28 | kỳ vọng 28
127 | kỳ vọng 127


Một chi tiết: khi nào "ghép khác bit được"? Khi `best(A0, B1, ...)` hoặc `best(A1, B0, ...)` trả giá trị `>= 0`, tức hai nhóm tương ứng **đều không rỗng** (ta dùng `-1` cho "không có cặp"). Khi `A` và `B` là cùng một nhóm (lúc đầu), trường hợp `A0` ghép `B1` chính là cặp `(số có bit 0, số có bit 1)`; còn cặp `(x, x)` được tính ở nhánh "cùng bit", cho XOR `0`.

In [17]:
# Code nộp LeetCode — Cách 4 (chạy đúng; đệ quy chỉ sâu 31 tầng, nhưng tốn nhiều list tạm nên chậm hơn Cách 2/3)
class Solution4:
    def findMaximumXOR(self, nums: List[int]) -> int:
        def best(A: List[int], B: List[int], bit: int) -> int:
            if not A or not B:
                return -1
            if bit < 0:
                return 0
            A0 = [x for x in A if not x >> bit & 1]
            A1 = [x for x in A if x >> bit & 1]
            B0 = [x for x in B if not x >> bit & 1]
            B1 = [x for x in B if x >> bit & 1]
            cross = max(best(A0, B1, bit - 1), best(A1, B0, bit - 1))
            if cross >= 0:
                return (1 << bit) + cross
            return max(best(A0, B0, bit - 1), best(A1, B1, bit - 1))

        return best(nums, nums, 30)


print(Solution4().findMaximumXOR([3, 10, 5, 25, 2, 8]))   # 28

28


**Độ phức tạp:** Time `O(31 * n)` (mỗi tầng tổng kích thước các nhóm không vượt quá vài lần `n`; trường hợp "khác bit" tách đôi cặp `(A, B)` thành hai cặp con rời nhau); Space `O(31 * n)` do các danh sách tạm (hoặc `O(n)` nếu giải phóng sớm).

**Bốn cách tóm tắt:** (1) vét cạn; (2) tham lam bit + tập tiền tố; (3) Trie tìm đối lập; (4) chia nhóm theo bit. Cách 2 và 3 cùng dựa trên "bit cao thắng", cách 4 là cùng ý nhưng triển khai bằng đệ quy trên hai nhóm.

## 8. TRACE từng bước

Ta trace **Cách 2** trên `nums = [3, 10, 5, 25, 2, 8]` chỉ với 5 bit (bit 4 xuống bit 0) cho gọn, vì mọi số `< 32` (các bit 30..5 đều bằng 0 nên `ans` ở đó vẫn bằng 0).

Bảng trạng thái:

| bit i | tiền tố `x >> i` (tập) | `ans` sau `<<= 1` | `cand = ans \| 1` | có cặp `p ^ q = cand`? | `ans` mới |
|---|---|---|---|---|---|
| 4 | {0, 1} | 0 | 1 | có (0 ^ 1) | 1 |
| 3 | {0, 1, 3} | 2 | 3 | có (0 ^ 3) | 3 |
| 2 | {0, 1, 2, 6} | 6 | 7 | có (1 ^ 6) | 7 |
| 1 | {1, 2, 4, 5, 12} | 14 | 15 | không | 14 |
| 0 | {2, 3, 5, 8, 10, 25} | 28 | 29 | không | 28 |

Hãy tự kiểm tra từng hàng rồi chạy hàm trace để đối chiếu (ở bit 1 và bit 0 không có cặp nào cho `cand`, nên `ans` giữ nguyên `14` rồi `28`).

In [18]:
def trace_prefix_set(nums, top):
    ans = 0
    for i in range(top, -1, -1):
        prefixes = {x >> i for x in nums}
        ans <<= 1
        cand = ans | 1
        pair = next(((p, p ^ cand) for p in sorted(prefixes) if (p ^ cand) in prefixes), None)
        print(f"bit {i}: tiền tố = {sorted(prefixes)}")
        print(f"        ans<<1 = {ans} ({bin(ans)}), cand = {cand} ({bin(cand)}), cặp thỏa = {pair}")
        if pair is not None:
            ans = cand
        print(f"        -> ans = {ans} ({bin(ans)})")
    return ans


assert trace_prefix_set([3, 10, 5, 25, 2, 8], 4) == 28

bit 4: tiền tố = [0, 1]
        ans<<1 = 0 (0b0), cand = 1 (0b1), cặp thỏa = (0, 1)
        -> ans = 1 (0b1)
bit 3: tiền tố = [0, 1, 3]
        ans<<1 = 2 (0b10), cand = 3 (0b11), cặp thỏa = (0, 3)
        -> ans = 3 (0b11)
bit 2: tiền tố = [0, 1, 2, 6]
        ans<<1 = 6 (0b110), cand = 7 (0b111), cặp thỏa = (1, 6)
        -> ans = 7 (0b111)
bit 1: tiền tố = [1, 2, 4, 5, 12]
        ans<<1 = 14 (0b1110), cand = 15 (0b1111), cặp thỏa = None
        -> ans = 14 (0b1110)
bit 0: tiền tố = [2, 3, 5, 8, 10, 25]
        ans<<1 = 28 (0b11100), cand = 29 (0b11101), cặp thỏa = None
        -> ans = 28 (0b11100)


Đọc output: ở bit 2, `cand = 7` và tập tiền tố có `{1, 6}`: `1 ^ 6 = 7` ✓ (`1 = 001` là tiền tố của `3` hay `2`; `6 = 110` là tiền tố của `25` — `25 >> 2 = 6`) ⇒ `ans = 7`. Cuối cùng `ans = 11100 = 28`, đạt bởi cặp `5` và `25`.

Bây giờ trace **Cách 3** (Trie): ta xây Trie chỉ với 5 bit và in đường đi của `search` cho từng số.

In [19]:
def trace_trie(nums, top):
    trie = Trie()
    ans = 0
    for x in nums:
        trie.insert(x, top)
        node, res, path = trie, 0, []
        for i in range(top, -1, -1):
            v = x >> i & 1
            if node.children[v ^ 1] is not None:
                res |= 1 << i
                node = node.children[v ^ 1]
                path.append(f"b{i}:muốn {v ^ 1} có")
            else:
                node = node.children[v]
                path.append(f"b{i}:muốn {v ^ 1} KHÔNG, đi {v}")
        ans = max(ans, res)
        print(f"x={x:>2} ({format(x, '05b')}): XOR tốt nhất với các số đã chèn = {res:>2} ({format(res, '05b')}) | ans = {ans}")
        print("      ", "; ".join(path))
    return ans


assert trace_trie([3, 10, 5, 25, 2, 8], 4) == 28

x= 3 (00011): XOR tốt nhất với các số đã chèn =  0 (00000) | ans = 0
       b4:muốn 1 KHÔNG, đi 0; b3:muốn 1 KHÔNG, đi 0; b2:muốn 1 KHÔNG, đi 0; b1:muốn 0 KHÔNG, đi 1; b0:muốn 0 KHÔNG, đi 1
x=10 (01010): XOR tốt nhất với các số đã chèn =  9 (01001) | ans = 9
       b4:muốn 1 KHÔNG, đi 0; b3:muốn 0 có; b2:muốn 1 KHÔNG, đi 0; b1:muốn 0 KHÔNG, đi 1; b0:muốn 1 có
x= 5 (00101): XOR tốt nhất với các số đã chèn = 15 (01111) | ans = 15
       b4:muốn 1 KHÔNG, đi 0; b3:muốn 1 có; b2:muốn 0 có; b1:muốn 1 có; b0:muốn 0 có
x=25 (11001): XOR tốt nhất với các số đã chèn = 28 (11100) | ans = 28
       b4:muốn 0 có; b3:muốn 0 có; b2:muốn 1 có; b1:muốn 1 KHÔNG, đi 0; b0:muốn 0 KHÔNG, đi 1
x= 2 (00010): XOR tốt nhất với các số đã chèn = 27 (11011) | ans = 28
       b4:muốn 1 có; b3:muốn 1 có; b2:muốn 1 KHÔNG, đi 0; b1:muốn 0 có; b0:muốn 1 có
x= 8 (01000): XOR tốt nhất với các số đã chèn = 17 (10001) | ans = 28
       b4:muốn 1 có; b3:muốn 0 KHÔNG, đi 1; b2:muốn 1 KHÔNG, đi 0; b1:muốn 1 KHÔNG, đi 0; b0:m

Quan sát: khi `x = 25` được xử lý, Trie đã chứa `3, 10, 5`; ở bit 4, `25` có bit `1` nên muốn nhánh `0` (có), ở bit 3 `25` có bit `1` muốn `0`... cuối cùng `XOR` tốt nhất là `28`. Các số `2, 8` đến sau không phá kỷ lục.

Trace **Cách 4**: in số phần tử mỗi nhóm ở từng tầng để thấy dữ liệu được chia nhỏ ra sao.

In [20]:
def trace_divide(nums, top):
    def best(A, B, bit, depth):
        pad = "  " * depth
        if not A or not B:
            return -1
        if bit < 0:
            return 0
        A0 = [x for x in A if not x >> bit & 1]
        A1 = [x for x in A if x >> bit & 1]
        B0 = [x for x in B if not x >> bit & 1]
        B1 = [x for x in B if x >> bit & 1]
        print(f"{pad}bit {bit}: A0={A0} A1={A1} | B0={B0} B1={B1}")
        c1 = best(A0, B1, bit - 1, depth + 1)
        c2 = best(A1, B0, bit - 1, depth + 1)
        cross = max(c1, c2)
        if cross >= 0:
            print(f"{pad}  -> ghép khác bit được, bit {bit} của XOR = 1, trả {(1 << bit) + cross}")
            return (1 << bit) + cross
        r = max(best(A0, B0, bit - 1, depth + 1), best(A1, B1, bit - 1, depth + 1))
        print(f"{pad}  -> chỉ ghép cùng bit, trả {r}")
        return r
    return best(nums, nums, top, 0)


print("kết quả:", trace_divide([3, 5, 25], 4))

bit 4: A0=[3, 5] A1=[25] | B0=[3, 5] B1=[25]
  bit 3: A0=[3, 5] A1=[] | B0=[] B1=[25]
    bit 2: A0=[3] A1=[5] | B0=[25] B1=[]
      bit 1: A0=[5] A1=[] | B0=[25] B1=[]
        bit 0: A0=[] A1=[5] | B0=[] B1=[25]
          -> chỉ ghép cùng bit, trả 0
        -> chỉ ghép cùng bit, trả 0
      -> ghép khác bit được, bit 2 của XOR = 1, trả 4
    -> ghép khác bit được, bit 3 của XOR = 1, trả 12
  bit 3: A0=[] A1=[25] | B0=[3, 5] B1=[]
    bit 2: A0=[25] A1=[] | B0=[3] B1=[5]
      bit 1: A0=[25] A1=[] | B0=[5] B1=[]
        bit 0: A0=[] A1=[25] | B0=[] B1=[5]
          -> chỉ ghép cùng bit, trả 0
        -> chỉ ghép cùng bit, trả 0
      -> ghép khác bit được, bit 2 của XOR = 1, trả 4
    -> ghép khác bit được, bit 3 của XOR = 1, trả 12
  -> ghép khác bit được, bit 4 của XOR = 1, trả 28
kết quả: 28


## 9. Code nộp LeetCode

Bản khuyến nghị: **Cách 2** (tham lam bit + tập tiền tố): 8 dòng, dễ nhớ, chạy nhanh trong Python vì thao tác trên `set` là mã C. Trie dùng nhiều đối tượng nên chậm hơn và tốn bộ nhớ.

In [21]:
class Solution:
    def findMaximumXOR(self, nums: List[int]) -> int:
        ans = 0
        for i in range(30, -1, -1):                  # 31 bit: bit 30 .. bit 0
            prefixes = {x >> i for x in nums}        # tiền tố gồm các bit 30..i của mọi số
            ans <<= 1                                # nới chỗ cho bit mới
            cand = ans | 1                           # thử đặt bit mới của đáp án bằng 1
            if any((p ^ cand) in prefixes for p in prefixes):
                ans = cand                           # tồn tại hai tiền tố XOR ra cand
        return ans


for nums, expected in examples:
    got = Solution().findMaximumXOR(nums)
    print(got, expected)
    assert got == expected
print(Solution().findMaximumXOR([5]), Solution().findMaximumXOR([2**31 - 1, 0]))   # 0  2147483647

28 28
127 127
0 2147483647


## 10. Vì sao đúng

**Định nghĩa.** Sau khi xử lý bit `i`, gọi `ans_i` là giá trị của biến `ans`. Khẳng định (bất biến): *`ans_i` bằng tiền tố (các bit `30..i`) của giá trị XOR **lớn nhất** có thể*.

**Chứng minh quy nạp theo bit giảm dần.**

- *Cơ sở:* trước bit 30, `ans = 0` (tiền tố rỗng) — đúng.
- *Bước:* giả sử `ans` (sau `<<= 1`) là tiền tố của đáp án tối ưu ở các bit `30..i+1`, ta xét bit `i`.
  - Gọi `M` là XOR lớn nhất thật sự. Nếu **tồn tại** cặp `(a, b)` mà tiền tố `a>>i ^ b>>i = cand` (= `ans` với bit `i` bằng 1) thì XOR `a ^ b` có tiền tố `cand`, mà `cand > ans` ở bit `i` nên mọi XOR có bit `i` = 0 (cùng các bit cao) nhỏ hơn `a ^ b`. Vậy bit `i` của `M` phải là `1` — đúng như thuật toán chọn (bit cao thắng).
  - Nếu **không tồn tại** cặp nào cho `cand`, mọi cặp có cùng tiền tố cao như `ans` đều có bit `i` bằng `0` ⇒ bit `i` của `M` là `0` ⇒ giữ `ans`.

  (Cặp đạt tiền tố `ans` ở các bit cao vẫn tồn tại vì bất biến.)

**Vì sao điều kiện `∃p, q: p ^ q = cand` ⇔ `∃p: (p ^ cand) ∈ prefixes`?** Đặt `q = p ^ cand`: `p ^ q = cand` ⇔ `q = p ^ cand` (tính chất đảo vế). Vậy chỉ cần kiểm tra `q` có trong tập.

**Vì sao XOR của hai tiền tố là tiền tố của XOR?** XOR hoạt động độc lập từng bit: các bit cao của `a ^ b` chỉ phụ thuộc các bit cao của `a` và `b`. Tức `(a ^ b) >> i = (a >> i) ^ (b >> i)`. Ta có thể kiểm chứng điều này bằng code ở dưới.

**Cặp `i == j`:** nếu `p = q` thì `p ^ q = 0`; `cand` luôn lẻ (bit thấp nhất bằng 1) nên `cand != 0`, vì thế không có chuyện nhầm "một phần tử ghép chính nó" tạo ra bit 1. Còn đáp án `0` (như `n = 1`) tự nhiên có được vì không bao giờ chọn được `cand`.

In [22]:
rng = random.Random(10)
for _ in range(1000):
    a, b, i = rng.randrange(2**31), rng.randrange(2**31), rng.randrange(31)
    assert (a ^ b) >> i == (a >> i) ^ (b >> i)
print("OK: (a ^ b) >> i == (a >> i) ^ (b >> i) trên 1000 mẫu")

OK: (a ^ b) >> i == (a >> i) ^ (b >> i) trên 1000 mẫu


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(n²)` | `O(1)` |
| 2. Tham lam bit + set | `O(31 n)` | `O(n)` |
| 3. Trie nhị phân | `O(31 n)` | `O(31 n)` |
| 4. Chia nhóm theo bit | `O(31 n)` | `O(31 n)` tạm |

Với `n = 2 * 10^5`: Cách 2 làm khoảng `31 * 2 * 10^5 * 2 = 1.2 * 10^7` thao tác trên set. Hãy đo thực tế.

In [23]:
rng = random.Random(2)
arr = [rng.randrange(2**31) for _ in range(200000)]
for name, f in [("Cách 2 set", max_xor_prefix_set)]:
    t = time.perf_counter()
    r = f(arr)
    print(f"{name:<12}: {r} trong {time.perf_counter() - t:.2f} giây (n = 2*10^5)")
small = arr[:20000]
for name, f in [("Cách 2 set", max_xor_prefix_set), ("Cách 3 trie", max_xor_trie), ("Cách 4 chia", max_xor_divide)]:
    t = time.perf_counter()
    r = f(small)
    print(f"{name:<12}: {r} trong {time.perf_counter() - t:.2f} giây (n = 20000)")

Cách 2 set  : 2147483647 trong 0.57 giây (n = 2*10^5)
Cách 2 set  : 2147483645 trong 0.05 giây (n = 20000)


Cách 3 trie : 2147483645 trong 0.49 giây (n = 20000)
Cách 4 chia : 2147483645 trong 0.14 giây (n = 20000)


## 12. Bộ test

Oracle độc lập: `max_xor_brute` (vét cạn mọi cặp) cho input nhỏ. Với input lớn ta đối chiếu chéo giữa Cách 2, 3, 4.

In [24]:
solutions = {
    "brute": max_xor_brute,
    "prefix-set": max_xor_prefix_set,
    "trie": max_xor_trie,
    "divide": max_xor_divide,
    "LeetCode Solution": lambda a: Solution().findMaximumXOR(a),
    "Solution2": lambda a: Solution2().findMaximumXOR(a),
    "Solution3": lambda a: Solution3().findMaximumXOR(a),
    "Solution4": lambda a: Solution4().findMaximumXOR(a),
}

# 12.1. Ví dụ đề
for nums, expected in examples:
    for name, f in solutions.items():
        assert f(nums) == expected, (name, nums)
print("12.1 OK")

12.1 OK


In [25]:
# 12.2. Edge case
edge = [
    ([0], 0),
    ([5], 0),                         # n = 1
    ([7, 7, 7], 0),                   # toàn bằng nhau
    ([0, 0], 0),
    ([1, 2], 3),
    ([2**31 - 1, 0], 2**31 - 1),      # cần bit 30
    ([2**31 - 1, 2**30], 2**30 - 1),  # 1111..1 ^ 1000..0
    ([2**30, 2**30 - 1], 2**31 - 1),
    ([3, 10, 5, 25, 2, 8], 28),
    ([8, 10, 2], 10),                 # 8 ^ 2 = 10
]
for nums, expected in edge:
    for name, f in solutions.items():
        got = f(nums)
        assert got == expected, (name, nums, got, expected)
print("12.2 OK:", len(edge), "edge case")

12.2 OK: 10 edge case


In [26]:
# 12.3. Ngẫu nhiên có seed, đối chiếu với vét cạn
rng = random.Random(421)
for _ in range(1500):
    n = rng.randint(1, 12)
    limit = rng.choice([4, 16, 1000, 2**31 - 1])
    nums = [rng.randint(0, limit) for _ in range(n)]
    expected = max_xor_brute(nums)
    for name, f in solutions.items():
        got = f(nums)
        assert got == expected, (name, nums, got, expected)
print("12.3 OK: 1500 test ngẫu nhiên, mọi cách khớp vét cạn")

12.3 OK: 1500 test ngẫu nhiên, mọi cách khớp vét cạn


In [27]:
# 12.4. Input lớn: đối chiếu chéo Cách 2 / 3 / 4 (n = 3000), số có nhiều bit chung
rng = random.Random(99)
for _ in range(5):
    base = rng.randrange(2**31)
    nums = [base ^ rng.randrange(1 << rng.randint(1, 31)) for _ in range(3000)]
    nums = [x & (2**31 - 1) for x in nums]
    a, b, c = max_xor_prefix_set(nums), max_xor_trie(nums), max_xor_divide(nums)
    assert a == b == c == max_xor_brute(nums)  # brute 3000^2/2 = 4.5M cặp, vẫn chấp nhận được
print("12.4 OK")

12.4 OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Duyệt thiếu bit: `range(30)` thay vì 31 bit

`nums[i]` có thể tới `2^31 - 1` có bit 30. `range(30)` chỉ cho bit `29..0` — bỏ mất bit cao nhất.

In [28]:
def wrong_missing_bit(nums):
    ans = 0
    for i in range(29, -1, -1):              # SAI: thiếu bit 30
        prefixes = {x >> i for x in nums}
        ans <<= 1
        cand = ans | 1
        if any((p ^ cand) in prefixes for p in prefixes):
            ans = cand
    return ans


nums = [2**30, 1]
print("sai  :", wrong_missing_bit(nums))
print("đúng :", max_xor_brute(nums))

sai  : 0
đúng : 1073741825


**Vì sao sai?** Tiền tố bây giờ là `x >> i` với `i <= 29`, nên `2^30` có "tiền tố" lớn hơn một bit so với `ans` ban đầu; thuật toán không còn tương ứng với "từng bit của đáp án". Sửa: `for i in range(30, -1, -1)`.

### Lỗi 2 — Quên `ans <<= 1` (hoặc cộng bit vào sai chỗ)

In [29]:
def wrong_no_shift(nums):
    ans = 0
    for i in range(30, -1, -1):
        prefixes = {x >> i for x in nums}
        cand = ans | 1                        # SAI: không dịch ans, cand luôn có bit thấp nhất = 1, ans không "lớn" lên
        if any((p ^ cand) in prefixes for p in prefixes):
            ans = cand
    return ans


rng = random.Random(5)
found = None
for _ in range(2000):
    nums = [rng.randint(0, 50) for _ in range(rng.randint(1, 5))]
    if wrong_no_shift(nums) != max_xor_brute(nums):
        found = nums
        break
print("phản ví dụ:", found, "| sai:", wrong_no_shift(found), "| đúng:", max_xor_brute(found))

phản ví dụ: [16, 47, 22, 50, 44] | sai: 1 | đúng: 63


**Vì sao sai?** Tiền tố `x >> i` có độ dài tăng dần theo từng vòng, nên `ans` cũng phải dài thêm một bit mỗi vòng (`ans <<= 1`). Không dịch thì `ans` và tiền tố lệch độ dài.

### Lỗi 3 — Trie: chọn cùng bit thay vì bit đối lập

In [30]:
def wrong_trie_same_bit(nums, top=30):
    trie = Trie()
    ans = 0
    for x in nums:
        trie.insert(x, top)
        node, res = trie, 0
        for i in range(top, -1, -1):
            v = x >> i & 1
            if node.children[v] is not None:        # SAI: ưu tiên nhánh CÙNG bit -> XOR ra 0
                node = node.children[v]
            else:
                res |= 1 << i
                node = node.children[v ^ 1]
        ans = max(ans, res)
    return ans


print("sai  :", wrong_trie_same_bit([3, 10, 5, 25, 2, 8]))
print("đúng :", max_xor_trie([3, 10, 5, 25, 2, 8]))

sai  : 0
đúng : 28


**Vì sao sai?** Ta muốn XOR **ra 1** ở bit cao nhất có thể ⇒ phải tìm số có bit **khác** `x`. Đi theo bit giống làm kết quả nhỏ nhất, không phải lớn nhất.

### Lỗi 4 — Nhầm với `i < j` bắt buộc và trả `-1`/`None` khi `n = 1`

Đề cho phép `i <= j`, nên `n = 1` có đáp án `0`. Nếu khởi tạo `best = -1` hoặc yêu cầu hai phần tử khác chỉ số, test `[5]` sẽ sai.

In [31]:
def wrong_strict_pairs(nums):
    best = -1                                   # SAI: coi 'i < j' và -1 nghĩa là không có cặp
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            best = max(best, nums[i] ^ nums[j])
    return best


print("sai  :", wrong_strict_pairs([5]), "(đề yêu cầu 0)")
print("đúng :", max_xor_brute([5]))

sai  : -1 (đề yêu cầu 0)
đúng : 0


**Bài học:** đọc kỹ `i <= j` — nó biến `n = 1` thành test hợp lệ với đáp án `0`.

## 14. Mẹo & nhận dạng pattern

- **"Tối đa XOR của hai số"** ⇒ Trie nhị phân hoặc tham lam từng bit với tập tiền tố.
- **Bit cao lấn át bit thấp** là mấu chốt của mọi bài "tối đa hóa theo bit": quyết định từ bit cao xuống thấp.
- **`a ^ b = c ⇔ a ^ c = b`**: biến bài toán "tìm cặp có XOR bằng `c`" thành "tra cứu `a ^ c` trong tập" (giống Two Sum với `^` thay `+`).
- Trie phù hợp khi có **truy vấn nhiều lần** (nhiều `x`, có thể kèm ràng buộc như `x <= limit`); tập tiền tố phù hợp cho bài tính một lần.
- Số tối đa `2^31 - 1` ⇒ **31 bit** (bit 30..0). Nếu `< 2^20` thì chỉ cần 20 bit — tối ưu hằng số bằng `max(nums).bit_length()`.
- Python: `set` và biểu thức sinh nhanh hơn dựng Trie nhiều đối tượng.

## 15. Biến thể

1. **Tối thiểu XOR của hai phần tử khác chỉ số:** sắp xếp rồi chỉ cần xét các cặp kề nhau (vì XOR nhỏ nhất đạt ở các số "gần nhau" nhất).
2. **Tối đa XOR với một số `x` cho trước (LeetCode 1707 - Maximum XOR With an Element From Array):** dùng Trie, thêm điều kiện `<= m` (sắp xếp truy vấn theo `m`, chèn dần).
3. **Đếm số cặp có XOR `< k` (LeetCode 1803):** Trie có đếm số phần tử trong mỗi nhánh.
4. **Tối đa XOR của hai số trong mảng con liên tiếp / tiền tố XOR:** dùng mảng tiền tố XOR rồi tối đa XOR hai tiền tố.
5. **Tối đa `a ^ b` có thêm ràng buộc `a != b`:** xử lý riêng nếu tất cả giống nhau.

In [32]:
# Biến thể 1: min XOR của hai phần tử ở hai vị trí khác nhau (n >= 2) bằng sort + cặp kề
def min_xor_pair(nums):
    s = sorted(nums)
    return min(s[i] ^ s[i + 1] for i in range(len(s) - 1))


def min_xor_pair_brute(nums):
    return min(nums[i] ^ nums[j] for i in range(len(nums)) for j in range(i + 1, len(nums)))


rng = random.Random(15)
for _ in range(500):
    nums = [rng.randint(0, 100) for _ in range(rng.randint(2, 10))]
    assert min_xor_pair(nums) == min_xor_pair_brute(nums)
print("biến thể 1:", min_xor_pair([3, 10, 5, 25, 2, 8]))     # 3^2 = 1

biến thể 1: 1


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Thử mọi cặp | `O(n²)` | `O(1)` | Làm oracle cho `n` nhỏ |
| 2. Tham lam bit + set | Quyết định bit cao nhất trước, tra tập tiền tố | `O(31 n)` | `O(n)` | **Khuyên dùng khi thi** (Python) |
| 3. Trie nhị phân | Tìm số "đối lập" tốt nhất cho từng số | `O(31 n)` | `O(31 n)` | Khi có nhiều truy vấn / thêm điều kiện |
| 4. Chia nhóm theo bit | Đệ quy trên hai nhóm A, B | `O(31 n)` | `O(31 n)` | Học cách nhìn bài dưới dạng "hai tập" |

Cả bốn cách cho cùng đáp án trên mọi test ở mục 12.

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `6 ^ 3`; (b) `format(10 ^ 5, "04b")`; (c) `25 >> 2` và `25 >> 2 & 1`; (d) `(9 ^ 14) ^ 9`.

**Bài 2 (tính tay).** Với `nums = [1, 2, 3]`, tính XOR của mọi cặp `i <= j` và giá trị lớn nhất. Sau đó thực hiện Cách 2 bằng tay với 2 bit (bit 1, bit 0).

**Bài 3 (điền TODO).** Hoàn thành `max_xor_todo(nums)` theo Cách 2.

In [33]:
def max_xor_todo(nums):
    # TODO 1: ans = 0
    # TODO 2: for i in range(30, -1, -1):
    # TODO 3: prefixes = {x >> i for x in nums}
    # TODO 4: ans <<= 1 ; cand = ans | 1
    # TODO 5: nếu tồn tại p trong prefixes với (p ^ cand) in prefixes thì ans = cand
    return None

# KHÔNG gọi hàm TODO ở đây để Run All không lỗi (TODO chưa điền).

**Bài 4 (tự viết lại).** Viết `max_xor_trie2(nums)` bằng Trie dùng **hai mảng** `left` và `right` (danh sách chỉ số nút con, `0` nghĩa là chưa có) **không nhìn lại** Cách 3. Kiểm tra với hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** Viết `best_pair(nums)` trả về `(a, b)` là một cặp giá trị đạt XOR lớn nhất (với `a <= b`). Kiểm tra: `[3,10,5,25,2,8] → (5, 25)`; `[7] → (7, 7)`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `6 ^ 3 = 110 ^ 011 = 101 = 5`; `10 ^ 5 = 1010 ^ 0101 = 1111`; `25 >> 2 = 6`, bit thấp nhất của 6 là `0`; `(a ^ b) ^ a = b`.
- **Mức 2 (vừa):** Bài 2: các cặp: `1^1=0, 1^2=3, 1^3=2, 2^2=0, 2^3=1, 3^3=0`, lớn nhất `3`. Bài 3: sao chép ý "tiền tố, `ans <<= 1`, `cand = ans | 1`, `any(...)`".
- **Mức 3 (sâu):** Bài 4: `left[node]`, `right[node]` là chỉ số nút con; thêm nút mới bằng `left.append(0); right.append(0)`. Bài 5: sau khi biết `M` (XOR lớn nhất), tìm cặp bằng tập: với mỗi `a`, nếu `a ^ M` có trong tập thì `(a, a ^ M)` là cặp cần tìm (đảo vế).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [34]:
# Bài 1
assert 6 ^ 3 == 5
assert format(10 ^ 5, "04b") == "1111"
assert (25 >> 2, 25 >> 2 & 1) == (6, 0)
assert (9 ^ 14) ^ 9 == 14

# Bài 2
nums2 = [1, 2, 3]
pairs = [nums2[i] ^ nums2[j] for i in range(3) for j in range(i, 3)]
assert pairs == [0, 3, 2, 0, 1, 0] and max(pairs) == 3


# Bài 3
def max_xor_todo_ans(nums):
    ans = 0
    for i in range(30, -1, -1):
        prefixes = {x >> i for x in nums}
        ans <<= 1
        cand = ans | 1
        if any((p ^ cand) in prefixes for p in prefixes):
            ans = cand
    return ans


for nums, expected in examples:
    assert max_xor_todo_ans(nums) == expected


# Bài 4
def max_xor_trie2(nums):
    left, right = [0], [0]                       # nút 0 là gốc; 0 trong left/right nghĩa là "chưa có con"
    ans = 0
    for x in nums:
        node = 0
        for i in range(30, -1, -1):              # chèn x
            v = x >> i & 1
            arr = right if v else left
            if arr[node] == 0:
                left.append(0)
                right.append(0)
                arr[node] = len(left) - 1
            node = arr[node]
        node, res = 0, 0
        for i in range(30, -1, -1):              # tìm bạn đối lập
            v = x >> i & 1
            want = left if v else right          # bit đối lập: nếu v = 1 thì muốn 0 (left)
            same = right if v else left
            if want[node]:
                res |= 1 << i
                node = want[node]
            else:
                node = same[node]
        ans = max(ans, res)
    return ans


for nums, expected in examples:
    assert max_xor_trie2(nums) == expected


# Bài 5
def best_pair(nums):
    M = max_xor_prefix_set(nums)
    s = set(nums)
    for a in nums:
        if (a ^ M) in s:
            return (min(a, a ^ M), max(a, a ^ M))


assert best_pair([3, 10, 5, 25, 2, 8]) == (5, 25)
assert best_pair([7]) == (7, 7)
rng = random.Random(17)
for _ in range(500):
    nums = [rng.randint(0, 200) for _ in range(rng.randint(1, 9))]
    a, b = best_pair(nums)
    assert a in nums and b in nums and a ^ b == max_xor_brute(nums)
    assert max_xor_trie2(nums) == max_xor_brute(nums)
print("OK: toàn bộ đáp án đúng")

OK: toàn bộ đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template** (tham lam bit + tập tiền tố)

```python
class Solution:
    def findMaximumXOR(self, nums: List[int]) -> int:
        ans = 0
        for i in range(30, -1, -1):                 # 31 bit
            prefixes = {x >> i for x in nums}
            ans <<= 1
            cand = ans | 1
            if any((p ^ cand) in prefixes for p in prefixes):
                ans = cand
        return ans
```

**Checklist**

- [ ] Duyệt **31 bit** (`range(30, -1, -1)`), không phải 30?
- [ ] `ans <<= 1` **trước** khi tạo `cand = ans | 1`?
- [ ] Dùng tập tiền tố `x >> i` và kiểm tra `p ^ cand` (đảo vế)?
- [ ] `n = 1` hoặc toàn số bằng nhau cho đáp án `0` (không phải `-1`)?
- [ ] Nếu dùng Trie: ưu tiên nhánh **đối lập** `v ^ 1`, và insert/search đủ 31 bit?
- [ ] Đã so sánh với vét cạn trên input nhỏ ngẫu nhiên, gồm cả số `2^31 - 1`?